In [ ]:
import os
import glob
import json
import random
import torch
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import MolsToGridImage
from IPython.display import display, HTML
from rdkit import RDLogger

# 屏蔽 RDKit 的所有警告信息，保持输出界面清爽
RDLogger.DisableLog('rdApp.*')

# =============================================================================
# 1. 路径配置 (确保在 CHEMCUT 工作目录下运行)
# =============================================================================
BASE_DIR = 'processed_data'
PT_DATA_DIR = os.path.join(BASE_DIR, 'SE3TD_Pre_datas_256')
VOCAB_FILE = os.path.join(BASE_DIR, 'fragment_vocabulary_256_pca.json')

# 原始 PDB/SDF 数据的根目录
RAW_DATA_DIR = os.path.join('datas', 'tain_datas', 'SE3TD', 'BioLiP', 'datas_BioLiP')

FEATURE_COLUMNS = [
    'avg_ecc', 'ecc_range', 'avg_logp', 'avg_tpsa', 'avg_charge', 
    'HBA', 'HBD', 'Aromatic', 'Hydrophobe', 'PosIonizable', 'NegIonizable',
    'max_dist_3d', 'max_angle_3d', 'max_plane_angle_3d'
]

NUM_SAMPLES = 5 

# =============================================================================
# 2. 加载基础字典
# =============================================================================
print("正在加载词汇表...")
with open(VOCAB_FILE, 'r', encoding='utf-8') as f:
    vocab_data = json.load(f)

vocab_smiles = list(vocab_data.keys())
vocab_vectors = torch.tensor([vocab_data[smi] for smi in vocab_smiles], dtype=torch.float32)

# =============================================================================
# 3. 扫描并随机抽取 .pt 文件
# =============================================================================
pt_files = glob.glob(os.path.join(PT_DATA_DIR, '**', '*.pt'), recursive=True)
if not pt_files:
    raise FileNotFoundError(f"未找到任何 .pt 文件，请检查路径！")

sample_files = random.sample(pt_files, min(NUM_SAMPLES, len(pt_files)))
print(f"共发现 {len(pt_files)} 个 .pt 文件，已随机抽取 {len(sample_files)} 个进行展示。\n")

# =============================================================================
# 4. 核心解析与展示逻辑
# =============================================================================
for file_path in sample_files:
    try:
        # 解除 weights_only 限制，允许加载 PyG Data 对象
        data = torch.load(file_path, weights_only=False)
    except Exception as e:
        print(f"读取文件 {file_path} 失败: {e}")
        continue
        
    pdb_id = getattr(data, 'pdb_id', 'Unknown')
    ligand_id = getattr(data, 'ligand_id', 'Unknown')
    
    display(HTML(f"<hr><h2 style='color: #2c3e50;'>复合物: {pdb_id} | 配体 ID: {ligand_id}</h2>"))
    
    # ---------------------------------------------------------
    # [修改点1] 加载并展示原始配体 (.sdf) 和 受体蛋白口袋 (.pdb)
    # ---------------------------------------------------------
    # 配体使用 .sdf 格式
    orig_ligand_path = os.path.join(RAW_DATA_DIR, str(pdb_id), f"{ligand_id}_ligand.sdf")
    # 口袋保持 .pdb 格式 (假设命名规范为 pdb_id_pocket.pdb，如与实际不符请微调)
    orig_pocket_path = os.path.join(RAW_DATA_DIR, str(pdb_id), f"{pdb_id}_pocket.pdb") 
    
    orig_ligand_mol = None
    orig_pocket_mol = None

    # 加载配体 (SDF)
    if os.path.exists(orig_ligand_path):
        try:
            orig_ligand_mol = Chem.MolFromMolFile(orig_ligand_path, sanitize=True)
            if orig_ligand_mol:
                AllChem.Compute2DCoords(orig_ligand_mol)
        except Exception:
            pass

    # 加载口袋 (PDB)
    if os.path.exists(orig_pocket_path):
        try:
            # 口袋较大，关闭 sanitize 防止解析过于严格导致失败
            orig_pocket_mol = Chem.MolFromPDBFile(orig_pocket_path, removeHs=True, sanitize=False)
            if orig_pocket_mol:
                AllChem.Compute2DCoords(orig_pocket_mol)
        except Exception:
            pass
    
    # 展示原始图像
    mols_to_show = []
    legends_to_show = []
    
    if orig_ligand_mol:
        mols_to_show.append(orig_ligand_mol)
        legends_to_show.append(f"Ligand: {ligand_id} (.sdf)")
    if orig_pocket_mol:
        mols_to_show.append(orig_pocket_mol)
        legends_to_show.append(f"Pocket: {pdb_id} (.pdb)")
        
    if mols_to_show:
        display(HTML(f"<b>1. 原始分子结构展示:</b>"))
        display(MolsToGridImage(mols_to_show, molsPerRow=2, subImgSize=(400, 400), legends=legends_to_show))
    else:
        display(HTML(f"<i>未能加载原始结构，请检查 {orig_ligand_path} 或 {orig_pocket_path} 是否存在且合法。</i>"))

    # ---------------------------------------------------------
    # 反推碎片 SMILES 与提取数据
    # ---------------------------------------------------------
    node_types = data.x[:, :2]
    is_ligand_mask = (node_types[:, 0] == 1.0)
    
    dists = torch.cdist(data.frag_embeds, vocab_vectors)
    best_match_indices = torch.argmin(dists, dim=1)
    matched_smiles_list = [vocab_smiles[idx.item()] for idx in best_match_indices]
    
    ligand_indices = torch.where(is_ligand_mask)[0].tolist()
    ligand_mols = []
    ligand_legends = []
    ligand_props_list = []
    
    for local_idx, global_idx in enumerate(ligand_indices):
        smi = matched_smiles_list[global_idx]
        mol = Chem.MolFromSmiles(smi) if smi != "<UNK>" else None
        ligand_mols.append(mol)
        legend_text = f"L-Frag {local_idx}"
        ligand_legends.append(legend_text)
        
        # [修改点3] 直接提取并使用原始数据，不再进行反归一化还原，仅保留4位小数提升可读性
        raw_feats = data.x[global_idx, 2:].tolist()
        raw_feats_rounded = [round(val, 4) for val in raw_feats]
        
        row_data = {"Fragment": legend_text, "SMILES": smi}
        row_data.update(dict(zip(FEATURE_COLUMNS, raw_feats_rounded)))
        ligand_props_list.append(row_data)

    pocket_indices = torch.where(~is_ligand_mask)[0].tolist()
    pocket_mols = []
    pocket_legends = []
    
    # 注意: 这里原来有一小段逻辑被省略了，补全口袋碎片的提取
    for local_idx, global_idx in enumerate(pocket_indices):
        smi = matched_smiles_list[global_idx]
        mol = Chem.MolFromSmiles(smi) if smi != "<UNK>" else None
        pocket_mols.append(mol)
        pocket_legends.append(f"P-Frag {local_idx}")

    # ---------------------------------------------------------
    # 展示碎片网格图
    # ---------------------------------------------------------
    valid_l_mols = [m for m in ligand_mols if m is not None]
    valid_l_legends = [l for m, l in zip(ligand_mols, ligand_legends) if m is not None]
    valid_p_mols = [m for m in pocket_mols if m is not None]
    valid_p_legends = [l for m, l in zip(pocket_mols, pocket_legends) if m is not None]
    
    if valid_l_mols:
        display(HTML("<b>2. 算法切割后的配体碎片 (Ligand Fragments):</b>"))
        display(MolsToGridImage(valid_l_mols, molsPerRow=5, subImgSize=(200, 200), legends=valid_l_legends))
    
    if valid_p_mols:
        display(HTML("<b>3. 受体口袋碎片样本 (Pocket Fragments, 截取前 10 个):</b>"))
        show_limit = min(10, len(valid_p_mols))
        display(MolsToGridImage(valid_p_mols[:show_limit], molsPerRow=5, subImgSize=(200, 200), legends=valid_p_legends[:show_limit]))

    # ---------------------------------------------------------
    # [修改点2] 展示配体内部边信息 (简洁的两列分类表)
    # ---------------------------------------------------------
    num_lig_frags = len(ligand_indices)
    
    if num_lig_frags > 0:
        global_to_local_lig = {global_idx: local_idx for local_idx, global_idx in enumerate(ligand_indices)}
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        
        edge_records = []
        
        for i in range(edge_index.size(1)):
            u = edge_index[0, i].item()
            v = edge_index[1, i].item()
            
            if u in global_to_local_lig and v in global_to_local_lig:
                local_u = global_to_local_lig[u]
                local_v = global_to_local_lig[v]
                
                # 因为是无向图，为了避免重复记录 A-B 和 B-A，规定只记录 local_u < local_v 的边
                if local_u < local_v:
                    attr_list = [int(x) for x in edge_attr[i].tolist()]
                    
                    # 尝试将独热向量解析为类型索引 (如 Type 0, Type 1)，解析失败则直接显示向量
                    try:
                        edge_type_idx = attr_list.index(1)
                        edge_desc = f"Type {edge_type_idx} {attr_list}"
                    except ValueError:
                        edge_desc = str(attr_list)
                        
                    edge_records.append({
                        "连接的碎片 (Node Pair)": f"L-Frag {local_u} ↔ L-Frag {local_v}",
                        "边属性 (Edge Attribute)": edge_desc
                    })
                    
        if edge_records:
            display(HTML("<b>4. 配体内部的拓扑连接关系 (简化表):</b>"))
            df_edges = pd.DataFrame(edge_records)
            
            styles = [
                dict(selector="th", props=[("text-align", "center"), ("background-color", "#f2f2f2")]),
                dict(selector="td", props=[("text-align", "center")])
            ]
            display(df_edges.style.set_table_styles(styles).set_table_attributes('style="font-size: 13px; width: 50%;"').hide(axis="index"))
        else:
            display(HTML("<i>4. 配体碎片之间无内部连接。</i>"))
    else:
        display(HTML("<i>4. 无配体碎片。</i>"))
        
    # ---------------------------------------------------------
    # 展示还原后的 14 维特征属性表 (直接展示原始数据)
    # ---------------------------------------------------------
    display(HTML("<b>5. 配体碎片化学特征 (原始模型数值):</b>"))
    df_props = pd.DataFrame(ligand_props_list)
    display(df_props.style.set_table_attributes('style="font-size: 12px"').hide(axis="index"))